# Lesson 08 Lab — Quantization Math: Scale, Zero Point, Group Size, and Error

**Puzzle:** Why does changing group size alter both model size and reconstruction error?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Uniform quantization stores integer codes plus scale metadata and, for asymmetric schemes, zero points. Granularity may be per tensor, row/channel, or group/block.

### Core mechanism

A common mapping is `q = clamp(round(x/s)+z, qmin, qmax)` and `x_hat = s(q-z)`. Symmetric INT4 typically uses `z=0` and a signed range near `[-8,7]`. Smaller groups estimate local ranges and reduce outlier sharing.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "08-quantization-math"
device = require_cuda()
torch.manual_seed(2026 + 8)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Smaller groups add scale loads and metadata and may miss a backend's supported block sizes. Larger groups are cheaper but one outlier can enlarge the step for many ordinary weights.

### What this code tests

The notebook holds the weight matrix fixed, changes only group size, and records both error and effective bits per weight.

**Experiment:** Quantize an outlier-containing matrix with INT4 group sizes 16, 64, and 128 and compare error plus metadata overhead.

**Declared evidence label:** `numerical-model`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
w = torch.randn(1024, 1024, device=device); w[:, ::97] *= 12
rows = []
for group in (16, 64, 128):
    q, scales, dq = symmetric_quantize(w, bits=4, group_size=group)
    metadata_bits = scales.numel() * 16
    rows.append({"group_size": group, "error": error_metrics(w, dq), "scale_count": scales.numel(),
                 "effective_bits_per_weight": round(4 + metadata_bits / w.numel(), 5),
                 "saturation_fraction": round((q.abs() == 7).float().mean().item(), 6)})
result = base_result(8, "numerical-model"); result.update({"shape": list(w.shape), "group_results": rows,
    "conclusion": "Smaller groups reduced local range sharing at the cost of more scale metadata."})


## 3. Inspect the evidence

Check saturation, error, and effective bits per value. Do not report the nominal four bits without scale overhead.

### Acceptance and rollback gate

Report nominal bits, scale/zero-point overhead, clipping rate, reconstruction error, group axis, and kernel-compatible group size together.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Smaller groups reduced local range sharing at the cost of more scale metadata.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "numerical-model",
  "executed_at_utc": "2026-08-07T14:45:26+00:00",
  "group_results": [
    {
      "effective_bits_per_weight": 5.0,
      "error": {
        "cosine": 0.99218798,
        "mae": 0.11072622,
        "max_abs": 2.84089684,
        "rmse": 0.20031616
      },
      "group_size": 16,
      "saturation_fraction": 0.077793,
      "scale_count": 65536
    },
    {
      "effective_bits_per_weight": 4.25,
      "error": {
        "cosine": 0.97160149,
        "mae": 0.25374436,
        "max_abs": 3.00970507,
        "rmse": 0.38436082
      },
      "group_size": 64,
      "saturation_fraction": 0.019199,
      "scale_count": 16384
    },
    {
      "effecti

## 4. Explain the result

Group size is an error–metadata–kernel compatibility decision, not a cosmetic configuration value.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).